# Getting Started with `molformula`

An interactive guide to chemical formula canonicalization, invariant-preserving formula arithmetic, ionization, and mass spectrometry applications (`[M+H]+`/`[M+Na]+`/`[M+HCOO]-` lipid adducts and phosphopeptide MS/MS product-ion series).

In [ ]:
%pip install -q molformula

from typing import Callable
from molformula import (
    Formula,
    FormulaChangeError,
    IonFormula,
    apply_formula_change,
    deprotonated,
    exact_mass,
    ion_mz,
    make_formula_change,
    protonated,
    sum_formulas,
    to_str,
)

## 1. Creating and Showing Canonical Formulas

`Formula` accepts molecular formulas and simple condensed formulas, canonicalizing output to Hill-system order. Explicit isotopes are written in brackets (e.g., `[13C]`), with the ordinary elemental symbol placed before explicit isotopes in increasing mass-number order.

In [ ]:
glucose = Formula('CH2OH(CHOH)4CHO')
to_str(glucose)

In [ ]:
labeled_glucose = Formula('[13C]H2OH(CHOH)4CHO')
to_str(labeled_glucose)

## 2. Formula Arithmetic

`apply_formula_change()` applies atom-species count gains and losses transactionally. Intermediate counts may be negative, provided the final `Formula` has non-negative counts and contains at least one atom.

In [ ]:
H2O = Formula('H2O')
apply_formula_change(glucose, losses=(H2O,))

In [ ]:
C, C13 = Formula('C'), Formula('[13C]')
apply_formula_change(glucose, gains=(C13,), losses=(C,))

## 3. Reusable Formula Changes

`make_formula_change()` bakes gains and losses into a reusable function that can be applied repeatedly without mutating precursor objects.

In [ ]:
dehydrated = make_formula_change(losses=(H2O,))
dehydrated(glucose)

In [ ]:
C13_labeled = make_formula_change(gains=(C13,), losses=(C,))
C13_labeled(glucose)

In [ ]:
C13_labeled(C13_labeled(glucose))

## 4. Atom-Species Count Validation (Acetaminophen Fragmentation)

`Formula` validates atom-species counts rather than chemical structure or reaction mechanisms. When a transformation would result in negative atom counts, `FormulaChangeError` is raised with the missing atom counts in `err.deficits`.

In [ ]:
acetaminophen = Formula('C8H9NO2')

ketene_lost = make_formula_change(losses=(Formula('C2H2O'),))
co2_lost = make_formula_change(losses=(Formula('CO2'),))

# Reported gas-phase MS/MS neutral loss (-C2H2O)
ketene_lost_acetaminophen = ketene_lost(acetaminophen)
ketene_lost_acetaminophen

In [ ]:
# Hypothetical CO2 loss: valid atom-species counts (two O atoms available)
co2_lost_acetaminophen = co2_lost(acetaminophen)
co2_lost_acetaminophen

In [ ]:
# Invalid atom-species counts: only one oxygen atom remains in C6H7NO
try:
    co2_lost(ketene_lost_acetaminophen)
except FormulaChangeError as err:
    print(f'Rejected CO2 loss from {ketene_lost_acetaminophen}; deficits: {err.deficits}')

## 5. Ionization, Exact Mass, and m/z

Ionization functions (`protonated`, `deprotonated`, `deelectronated`) construct `IonFormula` instances from `Formula` values treated as electrically neutral precursors. `exact_mass()` computes neutral exact mass, while `ion_mz()` accounts for the charge and electron mass.

In [ ]:
protonated_glucose = protonated(glucose, 1)
protonated_glucose

In [ ]:
deprotonated_glucose = deprotonated(glucose, 1)
deprotonated_glucose

In [ ]:
exact_mass(glucose)

In [ ]:
ion_mz(deprotonated_glucose)

## 6. Application 1: Operational Construction of Lipid Adduct Ions

Electrospray adduct ions can be constructed by composing neutral atom-species-count changes with ionization:
- `[M+H]+`: identity change + `protonated`
- `[M+Na]+`: `+Na -H` + `protonated`
- `[M+HCOO]-`: `+HCOOH` + `deprotonated`

In [ ]:
h2o = Formula('H2O')
glycerol = Formula('C3H8O3')
phosphocholine = Formula('C5H14NO4P')
sphingosine = Formula('C18H37NO2')
palmitic_acid = Formula('C16H32O2')
oleic_acid = Formula('C18H34O2')

pc_34_1 = apply_formula_change(
    glycerol,
    gains=(phosphocholine, palmitic_acid, oleic_acid),
    losses=(h2o, h2o, h2o),
)
sm_34_1 = apply_formula_change(
    sphingosine,
    gains=(phosphocholine, palmitic_acid),
    losses=(h2o, h2o),
)
cer_34_1 = apply_formula_change(
    sphingosine,
    gains=(palmitic_acid,),
    losses=(h2o,),
)

lipids = [pc_34_1, sm_34_1, cer_34_1]
lipids

In [ ]:
identity = make_formula_change()
sodium_substituted = make_formula_change(
    gains=(Formula('Na'),),
    losses=(Formula('H'),),
)
formic_acid_added = make_formula_change(
    gains=(Formula('HCOOH'),)
)

z = 1
[ion_mz(protonated(identity(l), z)) for l in lipids]

In [ ]:
[ion_mz(protonated(sodium_substituted(l), z)) for l in lipids]

In [ ]:
[ion_mz(deprotonated(formic_acid_added(l), z)) for l in lipids]

In [ ]:
def compose_ionization(
    formula_change: Callable[[Formula], Formula],
    ionization: Callable[[Formula, int], IonFormula],
    z: int,
) -> Callable[[Formula], IonFormula]:
    def _composed(f: Formula) -> IonFormula:
        return ionization(formula_change(f), z)

    return _composed

ion_transformations = [
    ('[M+H]+', compose_ionization(identity, protonated, z)),
    ('[M+Na]+', compose_ionization(sodium_substituted, protonated, z)),
    ('[M+HCOO]-', compose_ionization(formic_acid_added, deprotonated, z)),
]

for lipid in lipids:
    for label, transform in ion_transformations:
        ion = transform(lipid)
        print(f'{to_str(lipid)}, {label}: {to_str(ion.formula)} / {ion_mz(ion)}')

## 7. Application 2: Peptide Modification and b/y Product-Ion Calculations

Residue formulas can be combined with `sum_formulas()` and `protonated()` to construct $b$- and $y$-ion series across unmodified, phosphorylated (`+HPO3`), and neutral-loss (`-H3PO4`) states.
The same prefix/suffix approach can be adapted to other product-ion series.

For illustration, the H3PO4-loss atom-species counts of pSer are treated as a dehydroalanine residue in the conventional residue-list fragmentation model used below.


In [ ]:
def b_y_ion_series(
    seq: list[Formula],
    z: int,
) -> tuple[list[IonFormula], list[IonFormula]]:
    water = Formula('H2O')
    b_ions: list[IonFormula] = []
    y_ions: list[IonFormula] = []
    l = len(seq)
    for i in range(l):
        b_neutral = sum_formulas(*seq[: i + 1])
        b_ions.append(protonated(b_neutral, z))

        y_neutral = sum_formulas(water, *seq[l - 1 - i :])
        y_ions.append(protonated(y_neutral, z))

    return b_ions, y_ions

ala = Formula('HNCH(CH3)CO')
gly = Formula('HNCH2CO')
ser = Formula('HNCH(CH2OH)CO')
lys = Formula('HNCH(CH2CH2CH2CH2NH2)CO')

unmodified_seq = [ala, gly, ser, lys]
b_ions, y_ions = b_y_ion_series(unmodified_seq, 1)
[ion_mz(ion) for ion in b_ions]

In [ ]:
[ion_mz(ion) for ion in y_ions]

In [ ]:
phosphorylated = make_formula_change(gains=(Formula('HPO3'),))
h3po4_lost = make_formula_change(losses=(Formula('H3PO4'),))

pser = phosphorylated(ser)
h3po4_lost_pser = h3po4_lost(pser)
pser, h3po4_lost_pser

In [ ]:
def mzs_for_products(products: list[IonFormula]) -> list[float]:
    return [ion_mz(p) for p in products]

def print_product_mzs(product_mzs: list[float], product_type: str) -> None:
    for i, product_mz in enumerate(product_mzs):
        print(f'{product_type}{i+1}: {product_mz}')

phosphorylated_seq = [ala, gly, pser, lys]
h3po4_lost_seq = [ala, gly, h3po4_lost_pser, lys]

labeled_sequences = [
    ('Unmodified AGSK', unmodified_seq),
    ('AGpSK (Ser3 phosphorylated)', phosphorylated_seq),
    ('AGpSK after H3PO4 loss (hypothetical Dha residue)', h3po4_lost_seq),
]

for label, sequence in labeled_sequences:
    b_ions, y_ions = b_y_ion_series(sequence, 1)
    print(f'Product ions for {label}')
    print_product_mzs(mzs_for_products(b_ions), 'b')
    print_product_mzs(mzs_for_products(y_ions), 'y')
    print()